# DATA209 — Advanced Exploratory Data Analysis
# Practical P1-2 · Load and summarise

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 1 · Module 1 · CO1

---

**Objective.** Frame the problem, load the dataset, separate dependent from independent variables, and describe every numeric column by centre, spread and shape.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · P1-2 — Load and summarise

### Problem statement

An online retailer wants to understand **which browsing sessions end in a purchase**, so that
marketing spend can be directed at the sessions most likely to convert.

The data is one row per browsing session on an e-commerce site over a twelve-month period.
Before any predictive model is considered, we must establish what the data contains, whether
it can support the question, and what its variables actually look like.

**Dataset — Online Shoppers Purchasing Intention** (UCI Machine Learning Repository)
12,330 sessions x 18 columns; 10 numeric, 8 categorical or boolean.
This is the primary dataset for the whole lab manual.

In [ ]:
# ---- Load the dataset -------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError(
        "online_shoppers_intention.csv not found. Set DATA_DIR in the setup cell, "
        "or download it from the UCI repository."
    )

df = pd.read_csv(path)
print("Loaded:", path)
print("Shape :", df.shape, "->", f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# ---- Structure before statistics --------------------------------------
# Never compute a mean before you know the column's type is right.
structure = pd.DataFrame({
    "dtype"   : df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "nulls"   : df.isna().sum(),
    "unique"  : df.nunique(),
    "example" : df.iloc[0],
})
print(structure.to_string())
print("\nMemory:", round(df.memory_usage(deep=True).sum() / 1e6, 2), "MB")

### Dependent and independent variables

The **dependent variable** (target) is what we want to explain or predict.
The **independent variables** (features) are everything we might explain it with.

Getting this split right matters: the target must never be used as an input, and any variable
that would not be known at the moment of prediction must be excluded as well.

In [ ]:
# ---- Dependent vs independent ------------------------------------------
TARGET = "Revenue"          # True if the session ended in a purchase

y = df[TARGET]
X = df.drop(columns=[TARGET])

numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print(f"Dependent variable : {TARGET}  (dtype {y.dtype})")
print(f"Independent        : {X.shape[1]} variables")
print(f"  numeric      ({len(numeric_cols):2}): {numeric_cols}")
print(f"  categorical  ({len(categorical_cols):2}): {categorical_cols}")

print("\nTarget balance:")
print((y.value_counts(normalize=True) * 100).round(2).to_string())
print("\n-> 15.5% positive. Any model predicting 'no purchase' every time is 84.5% accurate.")
print("   Accuracy is therefore not an honest metric for this problem.")

### Mean vs. median

The mean is the balance point of the distribution; the median is the value that splits it in half.
They agree on symmetric data and diverge on skewed data. **The size of the gap is the diagnostic**:
a mean far above the median means a long right tail pulling the average up.

In [ ]:
# ---- Mean vs median ----------------------------------------------------
centre = pd.DataFrame({
    "mean"  : df[numeric_cols].mean(),
    "median": df[numeric_cols].median(),
})
centre["difference"]  = centre["mean"] - centre["median"]
# ratio guards against divide-by-zero on columns whose median is 0
centre["mean/median"] = np.where(centre["median"] != 0,
                                 centre["mean"] / centre["median"], np.nan)
centre = centre.sort_values("difference", ascending=False)
print(centre.to_string())

print("\nInterpretation")
print("- Where median = 0 the majority of sessions never touched that page type at all.")
print("- ProductRelated_Duration: mean far exceeds median -> strong right skew.")
print("- Report the median for these columns; the mean describes no typical session.")

### Variance vs. IQR

Variance and standard deviation measure spread around the *mean*, so they inherit its sensitivity
to extreme values. The IQR measures the width of the middle 50% and ignores the tails entirely.

Pair them correctly: **mean with standard deviation**, or **median with IQR**. Never mix.

In [ ]:
# ---- Variance vs IQR ---------------------------------------------------
q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)

spread = pd.DataFrame({
    "variance": df[numeric_cols].var(),
    "std"     : df[numeric_cols].std(),
    "IQR"     : q3 - q1,
    "range"   : df[numeric_cols].max() - df[numeric_cols].min(),
})
spread["std/IQR"] = np.where(spread["IQR"] != 0, spread["std"] / spread["IQR"], np.nan)
print(spread.sort_values("std/IQR", ascending=False).to_string())

print("\nInterpretation")
print("- std/IQR far above 1 means the tails are inflating the standard deviation.")
print("- Those columns need a robust summary (median + IQR) and, later, a transformation.")

### Skewness and distribution shape

Skewness quantifies horizontal asymmetry; kurtosis quantifies tail weight.
A working rule for reading the numbers:

| \|skew\| | Shape | What to do |
|---|---|---|
| < 0.5 | roughly symmetric | mean and standard deviation are fine |
| 0.5 – 1.0 | moderately skewed | prefer median; consider a mild transform |
| > 1.0 | strongly skewed | use median and IQR; transform before modelling |

In [ ]:
# ---- Skewness, kurtosis, shape classification --------------------------
def classify_shape(s):
    a = abs(s)
    if a < 0.5:  return "roughly symmetric"
    if a < 1.0:  return "moderately skewed"
    return "strongly skewed"

shape = pd.DataFrame({
    "skew"    : df[numeric_cols].skew(),
    "kurtosis": df[numeric_cols].kurtosis(),
    "zeros_%" : (df[numeric_cols] == 0).mean() * 100,
})
shape["direction"] = np.where(shape["skew"] > 0, "right tail", "left tail")
shape["shape"]     = shape["skew"].apply(classify_shape)
shape = shape.sort_values("skew", ascending=False)
print(shape.to_string())

strong = shape[shape["shape"] == "strongly skewed"].index.tolist()
print(f"\n{len(strong)} of {len(numeric_cols)} numeric columns are strongly skewed:")
print(" ", strong)
print("\nCarry this list to P23-24, where these columns are transformed.")

### Histogram, boxplot and density

Three views of the same variable, each hiding something the others reveal:

- **Histogram** — shows gaps and multiple peaks, but the story changes with bin width.
- **Boxplot** — compact five-number summary, ideal for comparing groups; hides multimodality completely.
- **Density (KDE)** — smooth shape, easy to overlay; the smoothness is invented by the bandwidth.

Always look at at least two.

In [ ]:
# ---- Histogram, boxplot, density for one variable ----------------------
col = "ProductRelated_Duration"

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
sns.histplot(df[col], bins=50, ax=axes[0], color="#3B6E8F")
axes[0].set_title(f"Histogram — {col}")
sns.boxplot(x=df[col], ax=axes[1], color="#3B6E8F")
axes[1].set_title(f"Boxplot — skew {df[col].skew():.2f}")
sns.kdeplot(df[col], ax=axes[2], fill=True, color="#3B6E8F")
axes[2].set_title("Density (KDE)")
plt.tight_layout(); plt.show()

print("Observe: the histogram is unreadable because a few very long sessions stretch the axis.")
print("The boxplot shows the same fact compactly. The KDE spills below zero, which is impossible")
print("for a duration — an artefact of smoothing, not a property of the data.")

In [ ]:
# ---- The same three views for every strongly skewed column -------------
show = strong[:6] if len(strong) >= 6 else strong
fig, axes = plt.subplots(len(show), 2, figsize=(11, 2.4 * len(show)))
axes = np.atleast_2d(axes)

for i, c in enumerate(show):
    sns.histplot(df[c], bins=40, ax=axes[i, 0], color="#3B6E8F")
    axes[i, 0].set_title(f"{c} — skew {df[c].skew():.2f}", loc="left")
    axes[i, 0].set_ylabel("")
    sns.boxplot(x=df[c], ax=axes[i, 1], color="#8B9199")
    axes[i, 1].set_title("")
    axes[i, 1].set_xlabel("")

plt.tight_layout(); plt.show()

### Pivot tables

A pivot table is non-graphical bivariate EDA: it summarises one variable across the levels of
others. Use it to quantify what a plot suggests, and to produce numbers you can put in a report.

In [ ]:
# ---- Pivot tables ------------------------------------------------------
# 1 — conversion rate by visitor type and weekend
pivot1 = pd.pivot_table(df, values="Revenue", index="VisitorType",
                        columns="Weekend", aggfunc="mean") * 100
print("Conversion rate (%) by visitor type and weekend")
print(pivot1.round(2).to_string(), "\n")

# 2 — median page value and session depth by month
pivot2 = pd.pivot_table(
    df, index="Month",
    values=["PageValues", "ProductRelated", "ProductRelated_Duration"],
    aggfunc="median")
print("Median engagement by month")
print(pivot2.round(2).to_string(), "\n")

# 3 — conversion and volume together: rate alone can mislead
pivot3 = df.groupby("Month").agg(
    sessions=("Revenue", "size"),
    purchases=("Revenue", "sum"),
    conversion_pct=("Revenue", lambda s: 100 * s.mean()),
).sort_values("conversion_pct", ascending=False)
print("Volume and conversion by month")
print(pivot3.round(2).to_string())

print("\nInterpretation")
print("- Always report the denominator. A high conversion rate on 200 sessions is weaker")
print("  evidence than a slightly lower rate on 3,000 sessions.")

### Deliverable — P1-2

One notebook, run top to bottom, containing:

1. The problem statement in your own words, with the sampling frame stated.
2. The dependent / independent split, justified.
3. A table of mean vs median and variance vs IQR for every numeric column.
4. The skewness table with each column classified by shape.
5. Histogram, boxplot and density for at least four variables.
6. At least two pivot tables.
7. **One sentence under every figure and table saying what it shows about the business question.**